# Working with Gold Schema Analytics 

| Step | Description |
| :--- | :--- |
| 1. **Read Gold Tables** | Loads all relevant dimension and fact tables from the `gold` schema into Spark DataFrames for analysis. |
| 2. **Trip Duration Calculation** | Calculates trip duration in minutes and formats it as hours:minutes:seconds for each trip. |
| 3. **Temporal Trip Analysis** | Analyzes trip frequency by hour, day of week, and month to uncover temporal ridership patterns. |
| 4. **Trip End Time Analysis** | Analyzes trip end times by hour, day of week, and month to understand when trips conclude. |
| 5. **Trip Duration Bucketing** | Buckets trips by duration (e.g., <1h, 1-3h, >24h) and counts trips in each bucket. |
| 6. **Trip Duration Statistics** | Computes average, median, minimum, and maximum trip durations in minutes. |
| 7. **Ride Duration by Rider Age** | Calculates average, median, and maximum ride duration grouped by rider age. |
| 8. **Trip and Station Statistics** | Computes total trip count, most common start/end stations, and trips that start and end at the same station. |
| 9. **Payment Amount Statistics** | Analyzes payment amounts by year, quarter, month, and rider age, including average, median, and maximum values. |

In [0]:
# Read all dim and fact tables from the gold schema
def read_gold_tables(table_names):
    return {name: spark.read.table(f"gold.{name}") for name in table_names}

tables = read_gold_tables([
    "fact_trip",
    "fact_payment",
    "dim_stations",
    "dim_riders",
    "dim_time"
])

df_fact_trip = tables["fact_trip"]
df_fact_payment = tables["fact_payment"]
df_dim_stations = tables["dim_stations"]
df_dim_riders = tables["dim_riders"]
df_dim_time = tables["dim_time"]

del tables

Silver Schema  

This section describes creating the Silver schema and storing files in the Silver layer.
The Silver layer typically contains cleaned and enriched data, structured for analytics.

In [0]:
from pyspark.sql import functions as F

df_trip_duration = df_fact_trip.withColumn(
    "start_at_ts", F.to_timestamp("start_at")
).withColumn(
    "ended_at_ts", F.to_timestamp("ended_at")
).withColumn(
    "trip_duration_minutes", ((F.col("ended_at_ts").cast("long") - F.col("start_at_ts").cast("long")) / 60).cast("int")
).withColumn(
    "trip_duration_hms",
    F.concat_ws(
        ":",
        (F.col("trip_duration_minutes") / 60).cast("int"),
        (F.col("trip_duration_minutes") % 60).cast("int"),
        F.lit("00")
    )
)

display(df_trip_duration.select(
    "trip_id", "start_at", "ended_at", "trip_duration_minutes", "trip_duration_hms"
))


trip_id,start_at,ended_at,trip_duration_minutes,trip_duration_hms
05961376528D5AA9,2021-06-22 11:35:53,2021-06-22 11:44:04,8,0:8:00
042029FFA981571B,2021-06-10 16:10:33,2021-06-10 16:26:43,16,0:16:00
F0E9906B452FC22A,2021-06-05 22:44:09,2021-06-05 23:04:56,20,0:20:00
D5548D6B05F9DB39,2021-06-18 18:24:07,2021-06-18 18:54:44,30,0:30:00
0CDF8C5894FDCAAB,2021-06-17 19:33:49,2021-06-17 19:42:28,8,0:8:00
4BFFBEC71C1DDAAB,2021-06-17 18:04:25,2021-06-17 18:22:40,18,0:18:00
28027CAC28015DD8,2021-06-21 14:03:16,2021-06-21 14:14:28,11,0:11:00
BD98F6308527AB12,2021-06-12 17:34:34,2021-06-12 17:42:20,7,0:7:00
04DB320B56DB1AED,2021-06-23 20:36:29,2021-06-23 20:46:34,10,0:10:00
FB07D72716C496F4,2021-06-06 23:47:50,2021-06-06 23:51:50,4,0:4:00


Silver Schema  

This section describes creating the Silver schema and storing files in the Silver layer.
The Silver layer typically contains cleaned and enriched data, structured for analytics.

In [0]:
# Analyze trip frequency by hour, day of week, and month to uncover temporal ridership patterns
df_trip_time_analysis = df_fact_trip.withColumn(
    "start_at_ts", F.to_timestamp("start_at")
).withColumn(
    "ride_hour", F.hour("start_at_ts")
).withColumn(
    "ride_day_of_week", F.date_format("start_at_ts", "EEEE")
).withColumn(
    "ride_month", F.month("start_at_ts")
)

display(df_trip_time_analysis.groupBy("ride_hour").count().orderBy("ride_hour"))
display(df_trip_time_analysis.groupBy("ride_day_of_week").count().orderBy("ride_day_of_week"))
display(df_trip_time_analysis.groupBy("ride_month").count().orderBy("ride_month"))

ride_hour,count
0,67744
1,47310
2,28751
3,15372
4,12719
5,34706
6,90930
7,166642
8,199719
9,169069


ride_day_of_week,count
Friday,653417
Monday,576105
Saturday,822470
Sunday,713526
Thursday,598289
Tuesday,604777
Wednesday,616336


ride_month,count
1,80128
2,42995
3,205691
4,298207
5,450994
6,608778
7,692321
8,674409
9,621150
10,477972


In [0]:
df_trip_end_time_analysis = df_fact_trip.withColumn(
    "ended_at_ts", F.to_timestamp("ended_at")
).withColumn(
    "end_hour", F.hour("ended_at_ts")
).withColumn(
    "end_day_of_week", F.date_format("ended_at_ts", "EEEE")
).withColumn(
    "end_month", F.month("ended_at_ts")
)

display(df_trip_end_time_analysis.groupBy("end_hour").count().orderBy("end_hour"))
display(df_trip_end_time_analysis.groupBy("end_day_of_week").count().orderBy("end_day_of_week"))
display(df_trip_end_time_analysis.groupBy("end_month").count().orderBy("end_month"))

end_hour,count
0,75427
1,52732
2,35179
3,18669
4,13510
5,29094
6,78059
7,149949
8,197535
9,164587


end_day_of_week,count
Friday,650202
Monday,577361
Saturday,819607
Sunday,719112
Thursday,597393
Tuesday,604795
Wednesday,616450


end_month,count
1,80180
2,42987
3,205673
4,298148
5,450924
6,608789
7,692007
8,674797
9,621147
10,478012


In [0]:
df_trip_duration_bucketed = df_trip_duration.withColumn(
    "trip_duration_hours", (F.col("trip_duration_minutes") / 60).cast("int")
).withColumn(
    "duration_bucket",
    F.when(F.col("trip_duration_hours") < 1, "<1h")
     .when((F.col("trip_duration_hours") >= 1) & (F.col("trip_duration_hours") < 24),
           F.concat(F.lit(F.col("trip_duration_hours") - F.col("trip_duration_hours") % 2), 
                    F.lit("-"), 
                    F.lit(F.col("trip_duration_hours") - F.col("trip_duration_hours") % 2 + 2), 
                    F.lit("h")))
     .otherwise(">24h")
)

df_duration_bucket_counts = df_trip_duration_bucketed.groupBy("duration_bucket").count().orderBy("duration_bucket")

display(df_duration_bucket_counts)

duration_bucket,count
0-2h,160725
10-12h,708
12-14h,609
14-16h,498
16-18h,457
18-20h,360
2-4h,43939
20-22h,343
22-24h,291
4-6h,4262


In [0]:
# Calculate average, median, min, and max ride duration in minutes
df_duration_stats = df_trip_duration.select("trip_duration_minutes").filter(F.col("trip_duration_minutes") > 0)

avg_duration = df_duration_stats.agg(F.avg("trip_duration_minutes").alias("avg_duration")).first()["avg_duration"]
min_duration = df_duration_stats.agg(F.min("trip_duration_minutes").alias("min_duration")).first()["min_duration"]
max_duration = df_duration_stats.agg(F.max("trip_duration_minutes").alias("max_duration")).first()["max_duration"]

median_duration = df_duration_stats.approxQuantile("trip_duration_minutes", [0.5], 0.01)[0]

stats_df = spark.createDataFrame(
    [(avg_duration, median_duration, min_duration, max_duration)],
    [
        "average_duration_minutes",
        "median_duration_minutes",
        "min_duration_minutes",
        "max_duration_minutes"
    ]
)

display(stats_df)

average_duration_minutes,median_duration_minutes,min_duration_minutes,max_duration_minutes
21.57880997912655,12.0,1,55944


Silver Schema  

This section describes creating the Silver schema and storing files in the Silver layer.
The Silver layer typically contains cleaned and enriched data, structured for analytics.

In [0]:
%sql
SELECT
  r.is_member,
  AVG(
    ROUND((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60)
  ) AS avg_ride_duration_minutes,
  MIN(
    ROUND((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60)
  ) AS min_ride_duration_minutes,
  MAX(
    ROUND((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60)
  ) AS max_ride_duration_minutes,
  PERCENTILE_CONT(0.5) WITHIN GROUP (
    ORDER BY ROUND((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60)
  ) AS median_ride_duration_minutes
FROM proj4.gold.fact_trip t
INNER JOIN proj4.gold.dim_riders r
  ON t.rider_id = r.rider_id
WHERE unix_timestamp(t.ended_at) > unix_timestamp(t.start_at)
GROUP BY r.is_member
ORDER BY r.is_member

is_member,avg_ride_duration_minutes,min_ride_duration_minutes,max_ride_duration_minutes,median_ride_duration_minutes
False,21.33224504992532,0.0,47777.0,12.0
True,21.9133605045533,0.0,55944.0,12.0


In [0]:
query = """
SELECT
  YEAR(CURRENT_DATE) - YEAR(r.birthday) AS age,
  ROUND(AVG((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60), 0) AS avg_ride_duration_minutes,
  ROUND(MAX((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60), 0) AS max_ride_duration_minutes,
  ROUND(percentile_approx((unix_timestamp(t.ended_at) - unix_timestamp(t.start_at)) / 60, 0.5), 0) AS median_ride_duration_minutes
FROM gold.fact_trip t
INNER JOIN gold.dim_riders r
  ON t.rider_id = r.rider_id
WHERE unix_timestamp(t.ended_at) > unix_timestamp(t.start_at)
GROUP BY age
ORDER BY age
"""

display(spark.sql(query))

age,avg_ride_duration_minutes,max_ride_duration_minutes,median_ride_duration_minutes
20,23.0,25730.0,12.0
21,22.0,55944.0,12.0
22,21.0,19238.0,12.0
23,22.0,47777.0,12.0
24,22.0,32229.0,12.0
25,21.0,32859.0,12.0
26,22.0,27571.0,12.0
27,23.0,30743.0,12.0
28,22.0,38685.0,12.0
29,22.0,28246.0,12.0


In [0]:
# Total number of trips from fact_trip table
query_total_trips = """
SELECT COUNT(*) AS total_trip_count
FROM gold.fact_trip
"""
display(spark.sql(query_total_trips))

# Starting station and trip count
query_start = """
SELECT
  s.name AS start_station_name,
  COUNT(*) AS trip_count
FROM gold.fact_trip t
INNER JOIN gold.dim_stations s ON t.start_station_id = s.station_id
GROUP BY s.name
ORDER BY trip_count DESC
"""
display(spark.sql(query_start))

# End station and trip count
query_end = """
SELECT
  s.name AS end_station_name,
  COUNT(*) AS trip_count
FROM gold.fact_trip t
INNER JOIN gold.dim_stations s ON t.end_station_id = s.station_id
GROUP BY s.name
ORDER BY trip_count DESC
"""
display(spark.sql(query_end))

# Common start and end station
query_common = """
SELECT
  s1.name AS start_station_name,
  s2.name AS end_station_name,
  COUNT(*) AS trip_count
FROM gold.fact_trip t
INNER JOIN gold.dim_stations s1 ON t.start_station_id = s1.station_id
INNER JOIN gold.dim_stations s2 ON t.end_station_id = s2.station_id
WHERE t.start_station_id = t.end_station_id
GROUP BY s1.name, s2.name
ORDER BY trip_count DESC
"""
display(spark.sql(query_common))

total_trip_count
4584920


start_station_name,trip_count
Streeter Dr & Grand Ave,80344
Lake Shore Dr & North Blvd,46380
Lake Shore Dr & Monroe St,44672
Michigan Ave & Oak St,42722
Wells St & Concord Ln,41604
Millennium Park,40505
Clark St & Elm St,39346
Wells St & Elm St,35955
Theater on the Lake,35704
Kingsbury St & Kinzie St,32422


end_station_name,trip_count
Streeter Dr & Grand Ave,81840
Lake Shore Dr & North Blvd,52641
Michigan Ave & Oak St,43435
Lake Shore Dr & Monroe St,43151
Wells St & Concord Ln,41947
Millennium Park,41766
Clark St & Elm St,38767
Theater on the Lake,36192
Wells St & Elm St,35843
Kingsbury St & Kinzie St,31960


start_station_name,end_station_name,trip_count
Streeter Dr & Grand Ave,Streeter Dr & Grand Ave,13037
Lake Shore Dr & Monroe St,Lake Shore Dr & Monroe St,8571
Michigan Ave & Oak St,Michigan Ave & Oak St,6665
Millennium Park,Millennium Park,6446
Lake Shore Dr & North Blvd,Lake Shore Dr & North Blvd,4215
Theater on the Lake,Theater on the Lake,3999
Montrose Harbor,Montrose Harbor,3768
Buckingham Fountain,Buckingham Fountain,3624
Indiana Ave & Roosevelt Rd,Indiana Ave & Roosevelt Rd,3524
Shedd Aquarium,Shedd Aquarium,3274


In [0]:
# Payment amount statistics: yearly, quarterly, monthly, and by rider age (ignoring year, quarter, month for 2-4)

# Yearly stats
query_yearly = """
SELECT
  YEAR(p.date) AS year,
  ROUND(AVG(p.amount), 2) AS avg_amount,
  ROUND(MAX(p.amount), 2) AS max_amount,
  ROUND(percentile_approx(p.amount, 0.5), 2) AS median_amount
FROM gold.fact_payment p
GROUP BY year
ORDER BY year
"""
display(spark.sql(query_yearly))

# Quarterly stats (ignore year)
query_quarterly = """
SELECT
  QUARTER(p.date) AS quarter,
  ROUND(AVG(p.amount), 2) AS avg_amount,
  ROUND(MAX(p.amount), 2) AS max_amount,
  ROUND(percentile_approx(p.amount, 0.5), 2) AS median_amount
FROM gold.fact_payment p
GROUP BY quarter
ORDER BY quarter
"""
display(spark.sql(query_quarterly))

# Monthly stats (ignore year and quarter)
query_monthly = """
SELECT
  MONTH(p.date) AS month,
  ROUND(AVG(p.amount), 2) AS avg_amount,
  ROUND(MAX(p.amount), 2) AS max_amount,
  ROUND(percentile_approx(p.amount, 0.5), 2) AS median_amount
FROM gold.fact_payment p
GROUP BY month
ORDER BY month
"""
display(spark.sql(query_monthly))

# Rider age stats (ignore year, quarter, month)
query_age = """
SELECT
  YEAR(CURRENT_DATE) - YEAR(r.birthday) AS rider_age,
  ROUND(AVG(p.amount), 2) AS avg_amount,
  ROUND(MAX(p.amount), 2) AS max_amount,
  ROUND(percentile_approx(p.amount, 0.5), 2) AS median_amount
FROM gold.fact_payment p
INNER JOIN gold.dim_riders r ON p.rider_id = r.rider_id
GROUP BY rider_age
ORDER BY rider_age
"""
display(spark.sql(query_age))

year,avg_amount,max_amount,median_amount
2013,10.03,9.96,9.0
2014,10.06,9.99,9.0
2015,10.06,9.99,9.0
2016,10.0,9.99,9.0
2017,9.99,9.99,9.0
2018,9.98,9.99,9.0
2019,9.99,9.99,9.0
2020,10.0,9.99,9.0
2021,9.99,9.99,9.0
2022,9.99,9.99,9.0


quarter,avg_amount,max_amount,median_amount
1,10.0,9.99,9.0
2,9.99,9.99,9.0
3,10.0,9.99,9.0
4,9.99,9.99,9.0


month,avg_amount,max_amount,median_amount
1,10.0,9.99,9.0
2,10.0,9.99,9.0
3,10.0,9.99,9.0
4,9.99,9.99,9.0
5,9.99,9.99,9.0
6,9.99,9.99,9.0
7,9.99,9.99,9.0
8,10.01,9.99,9.0
9,9.99,9.99,9.0
10,10.0,9.99,9.0


rider_age,avg_amount,max_amount,median_amount
20,10.01,9.99,9.0
21,10.09,9.99,9.0
22,9.99,9.99,9.0
23,10.03,9.99,9.0
24,9.99,9.99,9.0
25,10.01,9.99,9.0
26,9.95,9.99,9.0
27,10.01,9.99,9.0
28,10.02,9.99,9.0
29,10.06,9.99,9.0
